# SHIVA-LLM Objective 02 — Stage 2C: Authenticated Event Crosswalk
**Runtime: Python 3 / CPU; 8–12 GB RAM recommended.** Upload Objective 01 Stage 1 ZIP (Parquet), Objective 01 Stage 2 ZIP, and Objective 02 Stage 2 state ZIP. This notebook refuses to produce a verified crosswalk if sequence reconstruction differs. No learner-split reassignment or guessed joins.

In [ ]:
!pip -q install pandas pyarrow numpy

In [ ]:
from google.colab import files
uploaded=files.upload()
print('Uploaded:',list(uploaded))

In [ ]:
import zipfile,io,json,os,hashlib
from pathlib import Path
import numpy as np,pandas as pd
archives={}
for name,data in uploaded.items():
    if name.lower().endswith('.zip'):
        z=zipfile.ZipFile(io.BytesIO(data)); archives[name]=z
        print(name, z.namelist()[:20])
def select(required):
    matches=[(name,z) for name,z in archives.items() if all(any(p.endswith(x) for p in z.namelist()) for x in required)]
    if len(matches)!=1: raise ValueError(f'Expected exactly one ZIP with {required}; found {[x[0] for x in matches]}')
    return matches[0]
def member(z,suffix):
    matches=[x for x in z.namelist() if x.endswith(suffix)]
    if len(matches)!=1: raise ValueError(f'Expected one {suffix}: {matches}')
    return z.read(matches[0])
name1,z1=select(['train_interactions.parquet','val_interactions.parquet','test_interactions.parquet'])
name2,z2=select(['train_sequences.npz','val_sequences.npz','test_sequences.npz'])
name3,z3=select(['hyperkt_states_train.csv','hyperkt_states_val.csv','hyperkt_states_test.csv'])
print('Stage1',name1,'Stage2',name2,'Predictions',name3)

In [ ]:
manifest=json.loads(member(z2,'manifest.json'))
vocab=json.loads(member(z2,'skill_vocab.json'))
lookup={str(k):int(v) for k,v in vocab.items()}
MAX_LEN=200
out=[]; diagnostics={}
for split in ('train','val','test'):
    df=pd.read_parquet(io.BytesIO(member(z1,f'{split}_interactions.parquet')))
    assert {'user','skill','correct','order'}.issubset(df.columns)
    df=df.sort_values(['user','order'],kind='mergesort')
    seq=np.load(io.BytesIO(member(z2,f'{split}_sequences.npz')))
    expected=[]; win=0
    for uid,g in df.groupby('user',sort=False):
        skills=np.asarray([lookup.get(str(x).strip(),0) for x in g['skill']],dtype=np.int32)
        ans=g['correct'].to_numpy(dtype=np.int8)
        orders=g['order'].to_numpy()
        if len(g)<2: continue
        for start in range(1,len(g),MAX_LEN):
            end=min(start+MAX_LEN,len(g)); L=end-start
            for pos in range(L):
                j=start+pos
                expected.append((win,pos,str(uid),str(orders[j]),int(skills[j-1]),int(ans[j-1]),int(skills[j]),int(ans[j])))
            win+=1
    cross=pd.DataFrame(expected,columns=['window_index','position','user_id','order_id','history_skill','history_correct','target_skill','target_correct'])
    assert win==len(seq['mask']),(split,'window count mismatch',win,len(seq['mask']))
    assert len(cross)==int(seq['mask'].sum()),(split,'target count mismatch')
    actual=[]
    for wi in range(len(seq['mask'])):
        for pos in np.flatnonzero(seq['mask'][wi]):
            actual.append((wi,int(pos),str(seq['user_id'][wi]),int(seq['history_skill'][wi,pos]),int(seq['history_correct'][wi,pos]),int(seq['target_skill'][wi,pos]),int(seq['target_correct'][wi,pos])))
    got=pd.DataFrame(actual,columns=['window_index','position','user_id','history_skill','history_correct','target_skill','target_correct'])
    for c in got.columns:
        if not cross[c].reset_index(drop=True).equals(got[c].reset_index(drop=True)):
            raise AssertionError(f'{split}: reconstructed {c} differs from saved sequence — STOP')
    if cross.duplicated(['user_id','order_id']).any():
        raise AssertionError(f'{split}: (user_id, order_id) not unique; cannot authenticate event IDs')
    pred=pd.read_csv(io.BytesIO(member(z3,f'hyperkt_states_{split}.csv')))
    print(split,'prediction columns:',pred.columns.tolist())
    assert {'window_index','position','user_id'}.issubset(pred.columns)
    pred['user_id']=pred['user_id'].astype(str)
    assert not pred.duplicated(['window_index','position']).any(),f'{split}: duplicated prediction positions'
    merged=cross.merge(pred,on=['window_index','position','user_id'],how='left',validate='one_to_one',indicator=True,suffixes=('','_pred'))
    if not merged['_merge'].eq('both').all() or len(pred)!=len(cross):
        raise AssertionError(f'{split}: incomplete or extra prediction rows — STOP')
    merged=merged.drop(columns=['_merge'])
    if 'y' in merged.columns and not np.array_equal(merged['target_correct'].to_numpy(),merged['y'].to_numpy()):
        raise AssertionError(f'{split}: prediction label mismatch')
    if 'target_correct_pred' in merged.columns and not np.array_equal(merged['target_correct'].to_numpy(),merged['target_correct_pred'].to_numpy()):
        raise AssertionError(f'{split}: prediction label mismatch')
    merged.insert(0,'split',split)
    out.append(merged)
    diagnostics[split]={'events':len(merged),'learners':merged.user_id.nunique(),'windows':win,'all_sequences_match':True,'all_predictions_aligned':True}
all_df=pd.concat(out,ignore_index=True)
assert not all_df.duplicated(['user_id','order_id']).any()
sets=[set(x.user_id) for x in out]
assert not (sets[0]&sets[1] or sets[0]&sets[2] or sets[1]&sets[2])
print(json.dumps(diagnostics,indent=2))

In [ ]:
# This audit authenticates alignment, not causal correctness of the frozen model.
# To audit model leakage, independently review its causal attention masks and target input construction.
from google.colab import files
import zipfile
report={'status':'event_alignment_verified','stage1_archive':name1,'stage2_archive':name2,'predictions_archive':name3,'checks':diagnostics,'limitations':['No claim of complete model-level temporal leakage audit','Knowledge probabilities are not latent state vectors','Stage 1 Objective 02 splits are incompatible; use HyperKT split assignments']}
output='/content/objective02_stage2c_crosswalk.zip'
all_df.to_csv('/content/hyperkt_event_crosswalk.csv',index=False)
with open('/content/stage2c_audit.json','w') as f:json.dump(report,f,indent=2)
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    z.write('/content/hyperkt_event_crosswalk.csv',arcname='hyperkt_event_crosswalk.csv')
    z.write('/content/stage2c_audit.json',arcname='stage2c_audit.json')
print('Saved',output)
files.download(output)